# Insert UIT ViQuAD2 into PostgreSQL

Only answerable examples are inserted. Each question is paired with the context chunk that contains its answer.

In [1]:
from pathlib import Path

import pandas as pd

root_dir = Path.cwd().parent
data_path = root_dir / "data" / "General" / "UIT_ViQuAD2.parquet"
viquad2 = pd.read_parquet(data_path)
viquad2 = viquad2.loc[~viquad2["is_impossible"]].copy()

print(f"Answerable examples: {len(viquad2):,}")

Answerable examples: 19,238


In [2]:
from sqlalchemy import select

from database.models import GeneralModel
from database.sql_manager import SQL_Manager

id_prefix = "data_003"
source = "UIT ViQuAD2"
sql_mng = SQL_Manager()
sql_mng.create_general_model()

existing_ids = set(
    sql_mng.con.scalars(
        select(GeneralModel.data_id).where(GeneralModel.data_id.like(f"{id_prefix}_%"))
    )
)
print(f"Existing UIT ViQuAD2 records: {len(existing_ids):,}")

Existing UIT ViQuAD2 records: 0


In [3]:
CHUNK_SIZE = 1_200
CHUNK_OVERLAP = 150


def answer_chunks(context: str, answers: dict) -> list[tuple[int, str]]:
    """Return context chunks containing at least one annotated answer."""
    selected_chunks = {}

    for answer_text, answer_start in zip(answers["text"], answers["answer_start"]):
        if len(answer_text) > CHUNK_SIZE:
            raise ValueError("An answer is longer than CHUNK_SIZE.")

        start = max(0, answer_start - CHUNK_OVERLAP)
        start = min(start, max(0, len(context) - CHUNK_SIZE))
        end = min(start + CHUNK_SIZE, len(context))
        selected_chunks[start] = context[start:end]

    return list(selected_chunks.items())

In [4]:
inserted = 0
skipped = 0

try:
    for _, row in viquad2.iterrows():
        for chunk_index, (_, context_chunk) in enumerate(
            answer_chunks(row["context"], row["answers"])
        ):
            data_id = f"{id_prefix}_{row['uit_id']}_{chunk_index:03d}"
            if data_id in existing_ids:
                skipped += 1
                continue

            sql_mng.insert_general_model(
                GeneralModel(
                    data_id=data_id,
                    source=source,
                    title=row["title"],
                    topic=None,
                    anchor=row["question"],
                    positive=context_chunk,
                    hard_negative=None,
                )
            )
            inserted += 1

            if inserted % 1_000 == 0:
                sql_mng.con.commit()

    sql_mng.con.commit()
finally:
    sql_mng.close()

print(f"Inserted: {inserted:,}; skipped existing: {skipped:,}")

Inserted: 19,392; skipped existing: 0
